# `altquotationline` — alternate/optional quotation lines

Unity Catalog: `ingestion_framework_test.bid_data_exploration.altquotationline`

Expected: bidder-proposed alternates (a substitution/option, not part of the base priced BOQ). No equivalent in the current sample-data pipeline — worth understanding whether these should be excluded from comparison entirely or surfaced separately.

## Run 1 — initial exploration ✅ *(run in Databricks — awaiting results to document findings)*

In [0]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.altquotationline

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM ingestion_framework_test.bid_data_exploration.altquotationline

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.altquotationline LIMIT 20

## Run 2 — follow-up queries ⏳ *(pending — not yet run)*

First pass findings (full write-up in `databricks/FINDINGS.md`):
- **Same shape as `quotationline`, plus `ALTQUOTATIONLINEID` (PK) and `ALTQUOTLINEUID`** — the latter looks like it points back to a specific base `quotationline.QUOTATIONLINEID`, i.e. "this is an alternate FOR that specific line" — unconfirmed, query below.
- **This sample batch is routine MRO/stock stuff** (coffee, cardamom, tea, bearings, light fittings, tools), all under `ORGID=TRANSORG` — not a construction BOQ. `BOQITEMNUM` is null here too, same as `quotationline`'s sample — still no confirmed populated example.
- **`QL2` (varchar(30)) looks like the real technical-acceptance status field** — values seen: `QUOTED`, `TNA` (Technically Not Acceptable?). Same field/values appeared in `quotationline`'s sample too. Worth confirming the full value set.
- There's a separate `SERVICE` column (decimal, boolean-like, all `0` in the sample) distinct from `LINETYPE='SERVICE'` — meaning unclear, probably a flag for something else. Low priority.

### Does `ALTQUOTLINEUID` really point back to a base `quotationline` row?

In [ ]:
%sql
SELECT a.RFQNUM, a.VENDOR, a.ALTQUOTLINEUID, q.QUOTATIONLINEID, q.DESCRIPTION AS base_description, a.DESCRIPTION AS alt_description
FROM ingestion_framework_test.bid_data_exploration.altquotationline a
JOIN ingestion_framework_test.bid_data_exploration.quotationline q ON a.ALTQUOTLINEUID = q.QUOTATIONLINEID
LIMIT 20

### Full value set for `QL2` (the QUOTED/TNA-looking status field)

In [ ]:
%sql
SELECT QL2, COUNT(*) AS n FROM ingestion_framework_test.bid_data_exploration.altquotationline
GROUP BY QL2 ORDER BY n DESC

### Row count and how common are alternates, relative to `quotationline`?

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM ingestion_framework_test.bid_data_exploration.altquotationline) AS alt_line_count,
  (SELECT COUNT(*) FROM ingestion_framework_test.bid_data_exploration.quotationline) AS base_line_count

### Any populated `BOQITEMNUM` here either?

In [ ]:
%sql
SELECT RFQNUM, VENDOR, BOQITEMNUM, DESCRIPTION, LINETYPE
FROM ingestion_framework_test.bid_data_exploration.altquotationline
WHERE BOQITEMNUM IS NOT NULL
LIMIT 30

### Does D-111808 have any alternates offered?

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.altquotationline
WHERE RFQNUM LIKE 'D-111808%'
LIMIT 30

**Observations (updated after first real run):**
- `ALTQUOTLINEUID` is the likely FK back to a specific base `quotationline.QUOTATIONLINEID` — pending the join query above to confirm rows actually match.
- Sample batch here is general MRO/stock RFQs, not construction — consistent with `quotationline`'s sample, still no real BOQ example with `BOQITEMNUM` populated across either table.
- `QL2` looks like a real technical-acceptance status field (`QUOTED`/`TNA`) — worth confirming the full distinct value set.